# Regional (ROI) Dataframe

Aggregates plaque-level data from `data.parquet` to a per-subject × per-atlas-region
summary and saves the result as `roi_data.parquet`.

Each row represents one subject in one atlas region and contains:
- Plaque count and volume metrics
- Plaque diameter metrics
- Vessel-proximity fractions (inside / near / far)
- Signed-distance-transform (SDT) summary statistics
- Plaque density (`plaque_count / region_volume_mm3`)


In [1]:
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ── Analysis parameters (kept in sync with other notebooks) ──────────────────
VOL_THRESH_ML     = 1e-4          # discard implausibly large plaques
TAU_UM            = 5.0           # near-vessel boundary (µm)
VESSEL_BINS_UM    = [0, 10, 20, np.inf]
VESSEL_BIN_LABELS = ["<10 µm", "10\u201320 µm", ">20 µm"]

TREAT_ORDER = ["PBS", "Lecanemab"]
GENO_ORDER  = ["ApoE3", "ApoE4"]

GROUP_COLS = ["subject", "treatment", "genotype", "sex", "index", "name"]

DATASET_NAMES = ["early", "late"]


## 1 · Load and filter plaque data


In [2]:
# Data loading and processing happens per-dataset below.
# The LUT (atlas look-up table) is shared across datasets and loaded next.


## 2 · Load atlas look-up table


In [3]:
lut = pd.read_csv("tpl-ABAv3_seg-all_dseg.tsv", sep="\t")
print(f"LUT: {len(lut):,} regions")
lut.head()


LUT: 2,654 regions


,index,name,abbreviation,color,volume_mm3
0,0,left root,root-L,000000,701.337955
1,1,left Basic cell groups and regions,grey-L,BFDAE3,0.000000
2,2,left Cerebrum,CH-L,B0F0FF,0.000000
3,3,left Cerebral cortex,CTX-L,B0FFB8,0.000000
4,4,left Cortical plate,CTXpl-L,70FF70,0.000000


## 3 · Classify plaques by vessel proximity

Mirrors the `classify_plaques()` function in `vessel_spatial_analysis.ipynb`.


In [4]:
def classify_plaques(
    plaque_df,
    tau_um=TAU_UM,
    vessel_bins_um=VESSEL_BINS_UM,
    vessel_bin_labels=VESSEL_BIN_LABELS,
):
    """Add vessel-relation category and min-vessel-diameter estimate."""
    out = plaque_df.copy()
    out["vessel_relation"] = pd.cut(
        out["sdt_CD31_um"],
        bins=[-np.inf, 0, tau_um, np.inf],
        labels=[
            "inside vessel",
            f"near vessel (0\u2013{tau_um:g} \u00b5m)",
            f"far from vessel (>{tau_um:g} \u00b5m)",
        ],
        include_lowest=True,
        right=True,
    )
    out["min_vessel_diam_um"] = np.nan
    inside = out["sdt_CD31_um"] < 0
    out.loc[inside, "min_vessel_diam_um"] = (
        out.loc[inside, "equiv_diam_um"] - 2 * out.loc[inside, "sdt_CD31_um"]
    )
    out["vessel_diam_bin"] = pd.cut(
        out["min_vessel_diam_um"],
        bins=vessel_bins_um,
        labels=vessel_bin_labels,
        include_lowest=True,
    )
    return out




## 4 · Aggregate to per-subject × per-ROI summary

For each subject × atlas region we compute:
- **Plaque count** and **volume** (sum, mean, median)
- **Equivalent diameter** (mean, median)
- **Signed distance transform** to nearest vessel (mean, median)
- **Vessel-proximity fractions** (inside / near / far as a share of that ROI's plaques)


In [5]:
def build_roi_dataframe(plaque_df, group_cols=GROUP_COLS):
    """Aggregate plaque metrics per subject per atlas ROI."""
    grp = plaque_df.groupby(group_cols, observed=True)

    # ── Core metrics ─────────────────────────────────────────────────────────
    agg = grp.agg(
        plaque_count   =("nvoxels",        "size"),
        total_vol_ml   =("plaque_vol_ml",   "sum"),
        mean_vol_ml    =("plaque_vol_ml",   "mean"),
        median_vol_ml  =("plaque_vol_ml",   "median"),
        mean_diam_um   =("equiv_diam_um",   "mean"),
        median_diam_um =("equiv_diam_um",   "median"),
        total_vol_um3  =("plaque_vol_um3",  "sum"),
        mean_sdt_um    =("sdt_CD31_um",     "mean"),
        median_sdt_um  =("sdt_CD31_um",     "median"),
    ).reset_index()

    # ── Vessel-proximity fractions ────────────────────────────────────────────
    rel_counts = (
        plaque_df
        .groupby(group_cols + ["vessel_relation"], observed=True)
        .size()
        .rename("n_rel")
        .reset_index()
        .merge(agg[group_cols + ["plaque_count"]], on=group_cols)
    )
    rel_counts["frac"] = rel_counts["n_rel"] / rel_counts["plaque_count"]

    # Pivot to wide format so each category becomes its own column
    PROX_RENAME = {
        "inside vessel":                         "frac_inside_vessel",
        f"near vessel (0\u2013{TAU_UM:g} \u00b5m)": "frac_near_vessel",
        f"far from vessel (>{TAU_UM:g} \u00b5m)": "frac_far_vessel",
    }
    rel_wide = (
        rel_counts
        .pivot_table(index=group_cols, columns="vessel_relation", values="frac", fill_value=0.0)
        .rename(columns=PROX_RENAME)
        .reset_index()
    )
    rel_wide.columns.name = None

    return agg.merge(rel_wide, on=group_cols, how="left")




## 5 · Process each dataset and save ROI dataframes

For each dataset (`early` / `late`): filter, classify plaques, aggregate to ROI level,
merge region volumes, compute plaque density, and save to `roi_data_{name}.parquet`.


In [6]:
for name in DATASET_NAMES:
    # ── load and filter ──────────────────────────────────────────────────
    df_raw = pd.read_parquet(f"data_{name}.parquet")
    df = df_raw.loc[df_raw["plaque_vol_ml"] <= VOL_THRESH_ML].copy()
    df["treatment"] = pd.Categorical(df["treatment"], categories=TREAT_ORDER, ordered=True)
    df["genotype"]  = pd.Categorical(df["genotype"],  categories=GENO_ORDER,  ordered=True)
    print(f"\n=== Dataset: {name} ===")
    print(f"Retained {len(df):,} plaques from {df['subject'].nunique()} subjects "
          f"(removed {len(df_raw) - len(df):,} oversized plaques)")

    # ── classify plaques by vessel proximity ─────────────────────────────
    df = classify_plaques(df)

    # ── aggregate to per-subject × per-ROI summary ────────────────────────
    roi_df = build_roi_dataframe(df)

    # ── merge region volumes and compute plaque density ───────────────────
    roi_df = roi_df.merge(lut[["index", "volume_mm3"]], on="index", how="left")
    roi_df["plaque_density"] = roi_df["plaque_count"] / roi_df["volume_mm3"]
    roi_df["vol_density_ml"] = roi_df["total_vol_ml"]  / roi_df["volume_mm3"]

    # ── save ──────────────────────────────────────────────────────────────
    out_path = f"roi_data_{name}.parquet"
    roi_df.to_parquet(out_path)
    print(f"Saved {out_path}  —  "
          f"{len(roi_df):,} rows × {len(roi_df.columns)} columns")
